# Quickstart: functionalize a protein with mBuild

**You bring:** a protonated protein PDB file and a fragment (as SMILES, SDF, or PDB).
**You get:** a modified PDB file plus bond records that OpenFF and GMSO can consume.

> If your PDB has no hydrogens, protonate it once first (e.g. pdbfixer at pH 7).
> This folder already contains `1ubq_protonated.pdb` (ubiquitin).


In [1]:
from mbuild.biopolymers import Protein, prepare_fragment

protein = Protein("1ubq_protonated.pdb")
print(len(list(protein.residues())), "residues | net formal charge:",
      protein.net_formal_charge)


76 residues | net formal charge: 0


## 1. Prepare your fragment

Write your fragment as SMILES with a `*` marking where the bond forms
(the star becomes the leaving hydrogen). Charges in the SMILES are kept.
Here: an octanoyl group, starred at the carbonyl carbon.


In [2]:
fragment = prepare_fragment("*C(=O)CCCCCCC", "OCT")
print("bond site:", fragment.link_atoms)


bond site: {'1': 'C1'}


## 2. Attach it

Pick the protein site by residue number + atom name. The fragment already
knows its own bond site from the `*`. One hydrogen leaves each side; the
fragment is aligned and bonded. If it lands too close to the protein,
mBuild relaxes it automatically with the protein held fixed (generic
parameters, protein coordinates unchanged; opt out with `relax=False`).


In [3]:
record = protein.attach(fragment, resnum=63, atom_name="NZ", chain_id="A")
print("new bond:", record.residue1.name, record.atom1_name, "-",
      record.residue2.name, record.atom2_name)
print("leaving hydrogens:", record.leaving1, record.leaving2)


2026-08-28 14:08:59,366 - mbuild.biopolymers.protein - WARNING - 1 atoms of the attached fragment sit within 1.0 A of existing atoms (closest: 0.43 A). Relax the structure before simulating (e.g. relax_fragments(), which holds the protein fixed).


new bond: LYS NZ - OCT C1
leaving hydrogens: ('HZ1',) ('H1',)


## 3. Write the modified PDB + bond records

`save_pdb` writes a standards-conformant file (residues, chains, TER, CONECT).
`bond_records()` returns one neutral record per new bond — everything a
downstream loader needs to know about the modification.


In [4]:
protein.save_pdb("1ubq_octanoyl.pdb", overwrite=True)
records = protein.bond_records()
records


[{'residue_names': ('LYS', 'OCT'),
  'residue_numbers': (63, 77),
  'atom_names': ('NZ', 'C1'),
  'leaving_atoms': (['HZ1'], ['H1']),
  'bond_order': 1}]

## 4a. Ingest with OpenFF (needs `openff-pablo >= 0.2` + `openff-toolkit`)

The fragment is not a CCD residue, so give Pablo one named definition built
from the same SMILES, then pass the spec straight through.


In [5]:
from rdkit import Chem
from openff.toolkit import Molecule
from openff.pablo import STD_CCD_CACHE, ResidueDefinition, topology_from_pdb

# Build the definition from the SAME starred SMILES: replace the star
# with H exactly as prepare_fragment did, so atom order matches.
star = Chem.RWMol(Chem.MolFromSmiles("*C(=O)CCCCCCC"))
for atom in star.GetAtoms():
    if atom.GetAtomicNum() == 0:
        atom.SetAtomicNum(1)
mol = star.GetMol()
Chem.SanitizeMol(mol)
offmol = Molecule.from_rdkit(Chem.AddHs(mol), allow_undefined_stereo=True)
for atom, particle in zip(offmol.atoms, fragment.particles()):
    atom.name = particle.name

# Format the neutral record into pablo's with_crosslink vocabulary.
record = records[0]
spec = {
    "residues": list(record["residue_names"]),
    "linking_atoms": list(record["atom_names"]),
    "leaving_atoms": [list(side) for side in record["leaving_atoms"]],
    "bond_order": record["bond_order"],
}
library = STD_CCD_CACHE.with_(
    {"OCT": [ResidueDefinition.from_molecule(offmol, residue_name="OCT")]}
).with_crosslink(**spec)

topology = topology_from_pdb("1ubq_octanoyl.pdb", residue_library=library)
molecule = topology.molecule(0)
print(molecule.n_atoms, "atoms | net charge:", molecule.total_charge)


1254 atoms | net charge: 0.0 elementary_charge


From here, `ForceField(...).create_interchange(topology)` assigns parameters.

## 4b. Or hand off to RDKit / GMSO / ParmEd directly

The modified protein also exports as objects, with chemistry intact where the
format supports it.


In [6]:
rdmol = protein.to_rdkit()      # sanitized, formal charges, residue info
print("RDKit:", rdmol.GetNumAtoms(), "atoms, net charge",
      Chem.GetFormalCharge(rdmol))

gmso_top = protein.to_gmso()    # GMSO topology (residue names survive)
print("GMSO:", gmso_top.n_sites, "sites")

structure = protein.to_parmed() # ParmEd structure, one residue per residue
print("ParmEd:", len(structure.residues), "residues")


RDKit: 1254 atoms, net charge 0
GMSO: 1254 sites
ParmEd: 77 residues


## Where to go next

- **One polymer tethered at two protein sites**: label the sites
  `[*:1]...[*:2]` and call
  `attach_multi(protein, smiles, sites=...)` from this folder's `mbuild_extras.py`.
  The chain is rotated, stretched onto both sites, and relaxed with the
  protein fixed - automatically.
- **Manual relaxation**: `protein.relax_fragments()` minimizes all attached
  fragments with the protein held fixed, any time.
- **Multi-residue polymers, branched glycans, GLYCAM fragments, mixed force
  fields, MD**: see `mbuild_pablo_ptm_demo.ipynb` in this folder.
- **Escape hatch for custom placements**: `protein.add_port_at(...)` returns a
  real mBuild `Port` for use with `force_overlap`.
